# Getting the discusion about the laws

With this notebook we extract the "discusion en sala" from each law

In [10]:
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import time
import re

In [11]:
url = "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8446/"
section_key = "1-4"   # the xajaxargs[] value you captured

s = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": url,
    "Origin": "https://www.bcn.cl",
}

r_get = s.get(url, headers=headers, timeout=30)
print("GET status:", r_get.status_code)
print("Cookies after GET:", s.cookies.get_dict())

xajaxr = str(int(time.time() * 1000))  # ms timestamp like your capture

data = [
    ("xajax", "verTextoCompleto"),
    ("xajaxr", xajaxr),
    ("xajaxargs[]", section_key),
]

r_post = s.post(url, headers=headers, data=data, timeout=30)
print("POST status:", r_post.status_code)
print("POST content-type:", r_post.headers.get("Content-Type"))
print("First 200 chars:\n", r_post.text[:200])

root = ET.fromstring(r_post.text)   # parse XML

cmd_nodes = root.findall(".//cmd[@n='js']")
print("Number of <cmd n='js'>:", len(cmd_nodes))

target_js = None
for node in cmd_nodes:
    txt = node.text or ""
    if ".modal-body" in txt and ".html('" in txt:
        target_js = txt
        break

print("Found modal-body injector:", target_js is not None)
print("Preview JS:\n", (target_js[:200] if target_js else "NONE"))

m = re.search(r"\.modal-body'\)\.html\('(.*)'\);\s*$", target_js, flags=re.DOTALL)
if not m:
    raise ValueError("Could not extract modal HTML from JS command.")
raw_html_string = m.group(1)
print("Raw html string length:", len(raw_html_string))

modal_html = raw_html_string
modal_html = modal_html.replace("\\'", "'")
modal_html = modal_html.replace('\\"', '"')
modal_html = modal_html.replace("\\n", "\n")
modal_html = modal_html.replace("\\t", "\t")
modal_html = modal_html.replace("\\r", "\r")
modal_html = modal_html.replace("\\\\", "\\")  # do this last

print(modal_html[:300])

soup = BeautifulSoup(modal_html, "html.parser")

item = soup.select_one("div.item")
print("Found div.item:", item is not None)

# metadata in attributes
metadata = dict(item.attrs) if item else {}
print("Metadata keys:", list(metadata.keys())[:10])

# transcript paragraphs
paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
paragraphs = [p for p in paragraphs if p]  # drop empties
print("Num paragraphs:", len(paragraphs))
print("First 5 paragraphs:")
for p in paragraphs[:5]:
    print("-", p)

GET status: 200
Cookies after GET: {'PHPSESSID': '0fe4b8h04attr2nlpefv2kno34'}
POST status: 200
POST content-type: text/xml; charset="UTF-8"
First 200 chars:
 <?xml version="1.0" encoding="UTF-8" ?><xjx><cmd n="js">$('.modal-body').html('');</cmd><cmd n="js"><![CDATA[$('.modal-body').html('<div class="item" fecha="2024-04-03" parteDocumento="http://datos.bc
Number of <cmd n='js'>: 4
Found modal-body injector: True
Preview JS:
 $('.modal-body').html('');
Raw html string length: 0

Found div.item: False
Metadata keys: []
Num paragraphs: 0
First 5 paragraphs:


In [12]:
cmd_nodes = root.findall(".//cmd[@n='js']")

for i, node in enumerate(cmd_nodes):
    txt = node.text or ""
    print("\n--- CMD", i, "---")
    print("length:", len(txt))
    print("contains div.item:", "div class=\"item\"" in txt)
    print("preview:", txt[:120].replace("\n", "\\n"))
    


--- CMD 0 ---
length: 26
contains div.item: False
preview: $('.modal-body').html('');

--- CMD 1 ---
length: 74319
contains div.item: True
preview: $('.modal-body').html('<div class="item" fecha="2024-04-03" parteDocumento="http://datos.bcn.cl/recurso/cl/documento/704

--- CMD 2 ---
length: 114
contains div.item: False
preview: $('.modal-title').html('Historia  de la Ley N° 21.788<br> 1. Primer Trámite Constitucional: Cámara de Diputados');

--- CMD 3 ---
length: 25
contains div.item: False
preview: $('#btnTrigger').click();


In [13]:
target_js = None
for node in cmd_nodes:
    txt = node.text or ""
    if "div class=\"item\"" in txt and "$('.modal-body').html(" in txt:
        target_js = txt
        break

if target_js is None:
    raise ValueError("Didn't find the JS command that injects the transcript HTML.")

In [14]:
start_token = "$('.modal-body').html('"
start = target_js.find(start_token)
if start == -1:
    raise ValueError("Start token not found in target_js.")

start += len(start_token)

end = target_js.rfind("');")
if end == -1 or end <= start:
    raise ValueError("End token not found (or invalid) in target_js.")

raw_html_string = target_js[start:end]
print("Raw html string length:", len(raw_html_string))
print("Raw preview:", raw_html_string[:120])

Raw html string length: 74293
Raw preview: <div class="item" fecha="2024-04-03" parteDocumento="http://datos.bcn.cl/recurso/cl/documento/704544/seccion/akn704544-p


In [15]:
modal_html = raw_html_string

# Unescape backslash escapes used inside JS strings
modal_html = modal_html.replace("\\\\", "\\")   # do double backslashes first
modal_html = modal_html.replace("\\'", "'")
modal_html = modal_html.replace('\\"', '"')
modal_html = modal_html.replace("\\n", "\n")
modal_html = modal_html.replace("\\t", "\t")
modal_html = modal_html.replace("\\r", "\r")

print(modal_html[:200])

<div class="item" fecha="2024-04-03" parteDocumento="http://datos.bcn.cl/recurso/cl/documento/704544/seccion/akn704544-po1-ds61" refersTo="http://datos.bcn.cl/ontologies/bcn-bills#DiscusionGeneral" ti


In [16]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(modal_html, "html.parser")

item = soup.select_one("div.item")
print("Found div.item:", item is not None)

metadata = dict(item.attrs) if item else {}
print("Metadata keys:", list(metadata.keys()))

paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
paragraphs = [p for p in paragraphs if p]
print("Num paragraphs:", len(paragraphs))
print("First 5 paragraphs:")
for p in paragraphs[:5]:
    print("-", p)

Found div.item: True
Metadata keys: ['class', 'fecha', 'partedocumento', 'refersto', 'tipodocumento', 'tramitacion', 'uridocumento']
Num paragraphs: 323
First 5 paragraphs:
- Fecha 03 de abril, 2024. Diario de Sesión en Sesión 13. Legislatura 372. Discusión General. Se aprueba en general y particular.
- ESTABLECIMIENTO DE NORMAS SOBRE PROTECCIÓN Y FOMENTO DE  ARTESANÍA (PRIMER TRÁMITE CONSTITUCIONAL. BOLETÍN N° 16371-24)
- La señorita  CICARDINI ,   doña Daniella  (Vicepresidenta).-
- Corresponde tratar el proyecto de ley, en primer trámite constitucional, iniciado en mensaje, que dicta normas sobre protección y fomento de la artesanía.
- Para la discusión de este proyecto se otorgarán tres minutos a cada diputada y diputado que se inscriba para hacer uso de la palabra.
